In [6]:
import cv2
import pytesseract
import time

# Optional: Set tesseract path if not in PATH
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

def main():
    cam_index = 0
    cap = cv2.VideoCapture(cam_index, cv2.CAP_DSHOW)
    time.sleep(2)

    ret, img = cap.read()
    cap.release()

    if not ret:
        print("❌ Couldn't capture image from camera.")
        return

    # Save image for reference
    cv2.imwrite("captured.jpg", img)

    # Preprocessing
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    enhanced = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # OCR with bounding boxes
    data = pytesseract.image_to_data(thresh, output_type=pytesseract.Output.DICT)
    print("🔍 Text boxes detected:", len(data['text']))

    for i in range(len(data['text'])):
        if int(data['conf'][i]) > 30 and data['text'][i].strip() != "":
            (x, y, w, h) = (data['left'][i], data['top'][i], data['width'][i], data['height'][i])
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(img, data['text'][i], (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    cv2.imshow("Text Detection", img)
    print("✅ Displaying result with rectangles. Press 'q' to quit.")

    while True:
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    raw_text = pytesseract.image_to_string(thresh)
    print("\n📋 Final Detected Text:\n", raw_text.strip())

    cv2.destroyAllWindows()

main()


🔍 Text boxes detected: 5
✅ Displaying result with rectangles. Press 'q' to quit.

📋 Final Detected Text:
 
